In [1]:
!pip install duckdb

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.5/15.5 MB 31.1 MB/s  0:00:00 32.7 MB/s eta 0:00:01

[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


In [6]:
import duckdb

con = duckdb.connect('mid.duckdb')
con.sql("CREATE OR REPLACE TABLE midi  AS SELECT * FROM read_csv_auto('MIDI_5.01.csv')")
con.sql("CREATE OR REPLACE TABLE midip AS SELECT * FROM read_csv_auto('MIDIP_5.01.csv')")
con.sql("SHOW TABLES").df()

,name
0,midi
1,midip


In [7]:
con.sql("""
CREATE OR REPLACE VIEW midip_d AS
SELECT *,
       CASE WHEN stday > 0 AND stmon > 0
            THEN make_date(styear, stmon, stday) END AS date
FROM midip
""")

con.sql("""
CREATE OR REPLACE VIEW midi_d AS
SELECT *,
       CASE WHEN stday > 0 AND stmon > 0
            THEN make_date(styear, stmon, stday) END AS date
FROM midi
""")
print('ok')

ok


## Column reference — MID 5.01

### Shared keys
| Column | Meaning |
|---|---|
| `dispnum` | Dispute ID. Same number = same standoff. Joins all four MID files. |
| `ord_incidnum` | Incident ID, renumbered into time order in v5.01. **Use this**, not `incidnum`. |
| `incidnum` | Old incident ID from v5.0. Not chronological. Ignore. |
| `version` | Dataset version (5). |

### Dates — all files
| Column | Meaning |
|---|---|
| `stday` / `stmon` / `styear` | Start date, split across three columns. `-9` = missing. |
| `endday` / `endmon` / `endyear` | End date, same format. |

### MIDI — one row per incident
| Column | Meaning |
|---|---|
| `duration` | Length of the incident in days. `-9` if dates incomplete. |
| `tbi` | Days between this incident and the neighbouring one. **Unreliable** — on the first incident it reports the gap to the *next* one, not the previous. Compute gaps from dates instead. |
| `fatality` | Deaths, banded: 0=none, 1=1–25, 2=26–100, 3=101–250, 4=251–500, 5=501–999, 6=999+, -9=missing. |
| `fatalpre` | Exact death count if known. `-9` = missing. |
| `action` | What happened, 0–21. See scale below. |
| `hostlev` | The action collapsed to 1–5. See scale below. |
| `numa` | How many countries on side A. |

### MIDIP — one row per country per

In [9]:
con.sql("SELECT * FROM midip_d WHERE dispnum = 4137 ORDER BY ord_incidnum").df()

,dispnum,ord_incidnum,stabb,ccode,stday,stmon,styear,endday,endmon,endyear,...,sidea,fatality,fatalpre,action,hostlev,revtype1,revtype2,version,incidnum,date
0,4137,4137001,ALB,339,-9,2,1998,-9,2,1998,...,1,0,0,11,3,2,0,5,4137001,NaT
1,4137,4137001,YUG,345,-9,2,1998,-9,2,1998,...,0,0,0,0,1,0,0,5,4137001,NaT
2,4137,4137002,MAC,343,1,3,1998,-9,3,1998,...,1,0,0,11,3,2,0,5,4137002,1998-03-01
3,4137,4137002,YUG,345,1,3,1998,-9,3,1998,...,0,0,0,0,1,0,0,5,4137002,1998-03-01
4,4137,4137003,ALB,339,6,3,1998,-9,3,1998,...,1,0,0,8,3,2,0,5,4137003,1998-03-06
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
520,4137,4137097,YUG,345,25,5,1999,26,5,1999,...,0,-9,-9,17,4,2,0,5,4137099,1999-05-25
521,4137,4137098,ALB,339,28,5,1999,28,5,1999,...,1,0,0,7,3,2,0,5,4137100,1999-05-28
522,4137,4137098,YUG,345,28,5,1999,28,5,1999,...,0,0,0,0,1,0,0,5,4137100,1999-05-28
523,4137,4137099,ALB,339,5,6,1999,5,6,1999,...,1,0,0,0,1,0,0,5,4137101,1999-06-05


In [11]:
con.sql("SELECT * FROM midi").df()

,dispnum,ord_incidnum,stday,stmon,styear,endday,endmon,endyear,duration,tbi,fatality,fatalpre,action,hostlev,numa,version,incidnum
0,3551,3551001,16,7,1992,16,7,1992,1,129,0,0,7,3,13,5,3551001
1,3551,3551002,-9,1,1993,-9,1,1993,-9,-9,0,0,7,3,3,5,3551003
2,3551,3551003,14,1,1993,-9,1,1993,-9,16,0,0,7,3,1,5,3551004
3,3551,3551004,30,1,1993,30,1,1993,1,4,0,0,15,4,1,5,3551005
4,3551,3551005,3,2,1993,3,2,1993,1,25,0,0,15,4,1,5,3551006
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4477,4723,4723002,4,10,2012,4,10,2012,1,-9,0,0,7,3,1,5,4723002
4478,4724,4724001,10,4,2013,11,4,2013,2,0,0,0,8,3,1,5,4724001
4479,4724,4724002,10,4,2013,10,4,2013,1,-9,0,0,8,3,1,5,4724002
4480,4725,4725001,14,7,2014,14,7,2014,1,-9,0,0,8,3,1,5,4725001


In [14]:
con.sql("""
CREATE OR REPLACE VIEW midi_clean AS
SELECT
    dispnum,
    ord_incidnum,
    CASE WHEN stday  > 0 AND stmon  > 0 THEN make_date(styear, stmon,  stday)  END AS start_date,
    CASE WHEN endday > 0 AND endmon > 0 THEN make_date(endyear, endmon, endday) END AS end_date,
    NULLIF(duration, -9)  AS duration,
    NULLIF(fatality, -9)  AS fatality,
    NULLIF(fatalpre, -9)  AS fatalpre,
    NULLIF(action,   -9)  AS action,
    NULLIF(hostlev,  -9)  AS hostlev,
    numa,
    -- flags, so nothing is dropped silently
    (stday <= 0 OR stmon <= 0) AS missing_start_date,
    (hostlev = -9)             AS missing_hostlev
FROM midi
""")

con.sql("""
CREATE OR REPLACE VIEW midip_clean AS
SELECT
    dispnum,
    ord_incidnum,
    stabb,
    ccode,
    CASE WHEN stday > 0 AND stmon > 0 THEN make_date(styear, stmon, stday) END AS start_date,
    insidea,
    sidea,
    NULLIF(action,   -9) AS action,
    NULLIF(hostlev,  -9) AS hostlev,
    NULLIF(revtype1, -9) AS revtype1,
    NULLIF(revtype2, -9) AS revtype2,
    (stday <= 0 OR stmon <= 0) AS missing_start_date
FROM midip
""")
print('views created')

views created


In [15]:
con.sql("""
SELECT *
FROM midi_clean
""").df() #null values for '-9'

,dispnum,ord_incidnum,start_date,end_date,duration,fatality,fatalpre,action,hostlev,numa,missing_start_date,missing_hostlev
0,3551,3551001,1992-07-16,1992-07-16,1,0,0,7,3,13,False,False
1,3551,3551002,NaT,NaT,<NA>,0,0,7,3,3,True,False
2,3551,3551003,1993-01-14,NaT,<NA>,0,0,7,3,1,False,False
3,3551,3551004,1993-01-30,1993-01-30,1,0,0,15,4,1,False,False
4,3551,3551005,1993-02-03,1993-02-03,1,0,0,15,4,1,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...
4477,4723,4723002,2012-10-04,2012-10-04,1,0,0,7,3,1,False,False
4478,4724,4724001,2013-04-10,2013-04-11,2,0,0,8,3,1,False,False
4479,4724,4724002,2013-04-10,2013-04-10,1,0,0,8,3,1,False,False
4480,4725,4725001,2014-07-14,2014-07-14,1,0,0,8,3,1,False,False


In [17]:
#all country pairs (dyads)
con.sql("""
CREATE OR REPLACE VIEW pair_incidents AS
SELECT
    a.dispnum,
    a.ord_incidnum,
    a.start_date,
    a.ccode   AS actor,
    t.ccode   AS target,
    a.hostlev,
    a.dispnum || '_' || LEAST(a.ccode, t.ccode) || '_' || GREATEST(a.ccode, t.ccode) AS dyad,
    COUNT(*) OVER (PARTITION BY a.ord_incidnum) AS n_pairs_in_incident
FROM midip_clean a
JOIN midip_clean t ON a.ord_incidnum = t.ord_incidnum
WHERE a.insidea = 1 AND t.insidea = 0 AND a.ccode <> t.ccode
""")

con.sql("SELECT COUNT(*) AS n_rows, COUNT(DISTINCT dyad) AS n_dyads FROM pair_incidents").df()

,n_rows,n_dyads
0,5137,941


In [18]:
#apply the validity rules
con.sql("""
CREATE OR REPLACE TABLE cohort AS
WITH first_row AS (
  SELECT dyad, dispnum, start_date AS t0, hostlev AS init_sev,
         ROW_NUMBER() OVER (PARTITION BY dyad ORDER BY ord_incidnum) AS rn
  FROM pair_incidents QUALIFY rn = 1),

mp AS (SELECT dyad, MAX(n_pairs_in_incident) AS max_pairs
       FROM pair_incidents GROUP BY dyad),

base AS (SELECT f.*, mp.max_pairs FROM first_row f JOIN mp USING (dyad)),

wk AS (SELECT b.dyad,
         MAX(CASE WHEN p.hostlev >= 4 THEN 1 ELSE 0 END) AS force_wk,
         COUNT(*)                                        AS n_incidents_wk,
         MAX(DATE_DIFF('day', b.t0, p.start_date))       AS days_to_last_wk
       FROM base b JOIN pair_incidents p USING (dyad)
       WHERE p.start_date BETWEEN b.t0 AND b.t0 + INTERVAL 7 DAY
       GROUP BY b.dyad),

recip AS (SELECT b.dyad,
            MAX(CASE WHEN EXISTS (
                SELECT 1 FROM pair_incidents q
                WHERE q.dyad = p.dyad AND q.actor = p.target AND q.target = p.actor
                  AND q.start_date BETWEEN b.t0 AND b.t0 + INTERVAL 7 DAY)
              THEN 1 ELSE 0 END) AS reciprocated_wk
          FROM base b JOIN pair_incidents p USING (dyad)
          WHERE p.start_date BETWEEN b.t0 AND b.t0 + INTERVAL 7 DAY
          GROUP BY b.dyad),

outc AS (SELECT b.dyad, MAX(CASE WHEN p.hostlev >= 4 THEN 1 ELSE 0 END) AS escalated
         FROM base b JOIN pair_incidents p USING (dyad)
         WHERE p.start_date >  b.t0 + INTERVAL 7  DAY
           AND p.start_date <= b.t0 + INTERVAL 97 DAY
         GROUP BY b.dyad)

SELECT b.dyad, b.dispnum, b.t0, YEAR(b.t0) AS year, b.init_sev,
   CASE WHEN b.t0 IS NULL      THEN 'first_incident_undated'
        WHEN b.init_sev >= 4   THEN 'starts_at_force'
        WHEN wk.force_wk = 1   THEN 'force_within_landmark'
        ELSE 'eligible' END                        AS status,
   COALESCE(o.escalated, 0)                        AS escalated,
   wk.n_incidents_wk,
   wk.days_to_last_wk,
   COALESCE(r.reciprocated_wk, 0)                  AS reciprocated_wk,
   CASE WHEN b.max_pairs > 1 THEN 1 ELSE 0 END     AS multiparty
FROM base b
LEFT JOIN wk    USING (dyad)
LEFT JOIN recip r USING (dyad)
LEFT JOIN outc  o USING (dyad)
""")
print('cohort built')

cohort built


In [20]:
#breakdown of valid/invalid data
con.sql("SELECT status, COUNT(*) AS n FROM cohort GROUP BY status ORDER BY n DESC").df()

,status,n
0,eligible,508
1,starts_at_force,333
2,first_incident_undated,70
3,force_within_landmark,30


In [22]:
#headline count. Eligible cases vs escalations | Percent that escalated
con.sql("""
SELECT COUNT(*) AS cases, SUM(escalated) AS escalations,
       ROUND(AVG(escalated)*100, 1) AS pct
FROM cohort WHERE status = 'eligible'
""").df()

,cases,escalations,pct
0,508,35.0,6.9


In [24]:
#if the case is before 2004 we use it to train
#if case is 2004-2009 then we mark it as 'dev'
#otherwise it is "final"

con.sql("""
SELECT CASE WHEN year < 2004 THEN 'train'
            WHEN year < 2010 THEN 'dev'
            ELSE 'final' END AS split,
       COUNT(*) AS cases, SUM(escalated) AS escalations,
       ROUND(AVG(escalated)*100, 1) AS pct
FROM cohort WHERE status = 'eligible'
GROUP BY 1 ORDER BY 1
""").df()

,split,cases,escalations,pct
0,dev,81,8.0,9.9
1,final,117,6.0,5.1
2,train,310,21.0,6.8


In [25]:
#multi-country check
con.sql("""
SELECT multiparty, COUNT(*) AS cases, SUM(escalated) AS escalations,
       ROUND(AVG(escalated)*100, 1) AS pct
FROM cohort WHERE status = 'eligible'
GROUP BY 1 ORDER BY 1
""").df()

,multiparty,cases,escalations,pct
0,0,362,30.0,8.3
1,1,146,5.0,3.4


In [26]:
con.sql("COPY cohort TO 'cohort.csv' (HEADER, DELIMITER ',')")
con.sql("COPY (SELECT * FROM pair_incidents) TO 'pair_incidents.csv' (HEADER, DELIMITER ',')")
print('saved')

saved


Prove four things can actually be pulled out of the raw data: who was fighting whom, whether it started with a threat rather than force, when things happened, and whether force came later.
And check the disputes with lots of countries in them. If sixteen countries act against one, don't just assume all sixteen were really in a fight with it. Look at them.
Paragraph 3 — pick a paper and copy it before changing anything.
Find a published paper that used data you can also get. Redo their analysis exactly as they did it, and see if you get their numbers.
Only after that, redo it your way with the seven-day and ninety-day rules. Report those as two separate results. Don't blend them together.


In [27]:
#who is fighting who (show we can extract dyads)
con.sql("""
SELECT COUNT(DISTINCT dyad)  AS distinct_dyads,
       COUNT(DISTINCT actor) AS distinct_acting_countries
FROM pair_incidents
""").df()

,distinct_dyads,distinct_acting_countries
0,941,139


In [30]:
#273 dyads have both sides acting. We can see who hit whom
con.sql("""
SELECT COUNT(*) AS dyads_where_both_sides_acted FROM (
  SELECT dyad FROM pair_incidents p
  WHERE EXISTS (SELECT 1 FROM pair_incidents q
                WHERE q.dyad = p.dyad AND q.actor = p.target AND q.target = p.actor)
  GROUP BY dyad)
""").df()

,dyads_where_both_sides_acted
0,273


In [32]:
#2: threat
#3: show of force
#total 586 potential candidates
con.sql("""
WITH f AS (SELECT dyad, hostlev,
             ROW_NUMBER() OVER (PARTITION BY dyad ORDER BY ord_incidnum) rn
           FROM pair_incidents QUALIFY rn = 1)
SELECT hostlev AS first_hostlev, COUNT(*) AS dyads
FROM f GROUP BY 1 ORDER BY 1
""").df()

,first_hostlev,dyads
0,2,86
1,3,500
2,4,352
3,5,3


In [34]:
#4.4% of the data does not have usable dates
con.sql("""
SELECT COUNT(*) AS rows,
       SUM(missing_start_date::INT) AS no_usable_date,
       ROUND(100.0 * SUM(missing_start_date::INT) / COUNT(*), 1) AS pct_missing,
       MIN(start_date) AS earliest,
       MAX(start_date) AS latest
FROM midip_clean
""").df()

,rows,no_usable_date,pct_missing,earliest,latest
0,9619,420.0,4.4,1992-01-20,2014-12-31


In [37]:
#all dyads that has violence in it.
con.sql("""
WITH f AS (SELECT dyad, start_date AS t0, hostlev,
             ROW_NUMBER() OVER (PARTITION BY dyad ORDER BY ord_incidnum) rn
           FROM pair_incidents QUALIFY rn = 1)
SELECT COUNT(*) AS dyads_with_force_in_window FROM (
  SELECT f.dyad
  FROM f JOIN pair_incidents p USING (dyad)
  WHERE f.hostlev < 4 AND f.t0 IS NOT NULL
    AND p.start_date >  f.t0 + INTERVAL 7  DAY
    AND p.start_date <= f.t0 + INTERVAL 97 DAY
    AND p.hostlev >= 4
  GROUP BY 1)
""").df()

,dyads_with_force_in_window
0,46


In [41]:
con.sql("""
CREATE OR REPLACE VIEW pair_incidents AS
SELECT
    a.dispnum,
    a.ord_incidnum,
    a.start_date,
    a.ccode   AS actor,
    t.ccode   AS target,
    a.stabb   AS actor_abb,
    t.stabb   AS target_abb,
    a.hostlev,
    a.dispnum || '_' || LEAST(a.ccode, t.ccode) || '_' || GREATEST(a.ccode, t.ccode) AS dyad,
    COUNT(*) OVER (PARTITION BY a.ord_incidnum) AS n_pairs_in_incident
FROM midip_clean a
JOIN midip_clean t ON a.ord_incidnum = t.ord_incidnum
WHERE a.insidea = 1 AND t.insidea = 0 AND a.ccode <> t.ccode
""")

In [43]:
con.sql(
    """
    SELECT * FROM pair_incidents
    """
).df()

,dispnum,ord_incidnum,start_date,actor,target,actor_abb,target_abb,hostlev,dyad,n_pairs_in_incident
0,3552,3552008,1993-01-07,690,645,KUW,IRQ,3,3552_645_690,1
1,3561,3561026,1993-10-26,365,372,RUS,GRG,3,3561_365_372,1
2,3564,3564023,1993-04-17,371,640,ARM,TUR,2,3564_371_640,1
3,4001,4001009,1995-02-28,811,800,CAM,THI,4,4001_800_811,1
4,4001,4001022,1996-02-22,800,811,THI,CAM,3,4001_800_811,1
...,...,...,...,...,...,...,...,...,...,...
5132,4706,4706007,2014-03-31,732,731,ROK,PRK,3,4706_731_732,2
5133,4706,4706007,2014-03-31,2,731,USA,PRK,3,4706_2_731,2
5134,4706,4706014,2014-09-19,731,732,PRK,ROK,3,4706_731_732,1
5135,4706,4706020,2014-10-18,731,732,PRK,ROK,3,4706_731_732,1


In [44]:
#how many pairs does each incident produce?
con.sql("""
SELECT n_pairs_in_incident AS pairs_made,
       COUNT(DISTINCT ord_incidnum) AS incidents,
       COUNT(*) AS pair_rows
FROM pair_incidents
GROUP BY 1 ORDER BY 1 DESC LIMIT 8
""").df()

,pairs_made,incidents,pair_rows
0,29,1,29
1,19,4,76
2,16,12,192
3,15,2,30
4,13,4,52
5,11,1,11
6,10,2,20
7,9,2,18


In [46]:
#NATO vs Libya counted as multiple
con.sql("""
SELECT dispnum, ord_incidnum, start_date, target_abb,
       COUNT(*) AS actors,
       STRING_AGG(actor_abb, ' ' ORDER BY actor_abb) AS acting_countries
FROM pair_incidents
GROUP BY 1,2,3,4
ORDER BY actors DESC LIMIT 3
""").df()

,dispnum,ord_incidnum,start_date,target_abb,actors,acting_countries
0,4685,4685007,2011-07-04,LIB,29,ALB BEL BUL CAN CRO CZR DEN EST FRN GMY GRC HU...
1,4137,4137068,1999-03-16,YUG,19,BEL CAN CZR DEN FRN GMY GRC HUN ICE ITA LUX NO...
2,4137,4137022,1998-08-17,YUG,19,ALB BEL CAN DEN FRN GMY GRC ICE ITA LIT LUX NO...


In [47]:
con.sql("""
SELECT CASE WHEN min_pairs > 3 THEN 'only ever in big incidents'
            ELSE 'appears in a small incident' END AS kind,
       COUNT(*) AS dyads
FROM (SELECT dyad, MIN(n_pairs_in_incident) AS min_pairs
      FROM pair_incidents GROUP BY dyad)
GROUP BY 1
""").df()

,kind,dyads
0,only ever in big incidents,91
1,appears in a small incident,850


In [48]:
#Not one of those pairs ever shows the target hitting back at that specific country. Among pairs that appear in small incidents, 32% do.
con.sql("""
WITH sz AS (SELECT dyad, MIN(n_pairs_in_incident) AS min_pairs
            FROM pair_incidents GROUP BY dyad),
     bd AS (SELECT DISTINCT p.dyad FROM pair_incidents p
            WHERE EXISTS (SELECT 1 FROM pair_incidents q
                          WHERE q.dyad = p.dyad AND q.actor = p.target AND q.target = p.actor))
SELECT CASE WHEN sz.min_pairs > 3 THEN 'only big incidents' ELSE 'has a small incident' END AS kind,
       COUNT(*) AS dyads,
       SUM(CASE WHEN bd.dyad IS NOT NULL THEN 1 ELSE 0 END) AS both_directions,
       ROUND(100.0 * SUM(CASE WHEN bd.dyad IS NOT NULL THEN 1 ELSE 0 END) / COUNT(*), 1) AS pct
FROM sz LEFT JOIN bd USING (dyad)
GROUP BY 1
""").df()

,kind,dyads,both_directions,pct
0,only big incidents,91,0.0,0.0
1,has a small incident,850,273.0,32.1
